# Multilingual Cyberbullying Detection

This notebook implements the research project's multitask architecture for multilingual English, Hindi, and Hinglish text. The pipeline combines XLM-RoBERTa with LoRA, a bidirectional GRU, Emotion-Aware Attention, and four task-specific heads for cyberbullying, emotion, sentiment, and sarcasm classification.

**Reproducibility note:** the raw datasets and trained checkpoint are not included in this public repository. See `data/README.md` for dataset handling information.

# Environment setup

Install the dependencies listed in `requirements.txt` before running the notebook.

For Google Colab, run the following in a cell if needed:

```bash
!pip install -r requirements.txt
```

The repository intentionally does not include the raw datasets or trained model checkpoint. See `data/README.md` for dataset instructions.

In [ ]:
# ============================================================
# CELL 2: Imports, Seeds, Device, Config
# ============================================================

import os, re, random, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW

from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
    precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

import nltk
import emoji
nltk.download('punkt', quiet=True)

# ── Seed ─────────────────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
set_seed()

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Config ───────────────────────────────────────────────────
# Repository paths
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
SAVE_DIR = OUTPUT_DIR / "saved_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

HATE_PATH    = DATA_DIR / "combined_hate_speech_dataset.csv"
EMOTION_PATH = DATA_DIR / "mlt_hinghlish_dataset.csv"

BASE_MODEL    = "xlm-roberta-base"
MAX_LEN       = 128
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.1
GRU_HIDDEN    = 256
GRU_LAYERS    = 2
BATCH_SIZE    = 16
EPOCHS        = 10
LR_LORA       = 3e-5
LR_HEAD       = 1.5e-4
WEIGHT_DECAY  = 0.01
MAX_GRAD_NORM = 1.0
FP16          = True

NUM_CYBER     = 2
NUM_EMOTION   = 10
NUM_SENTIMENT = 3
NUM_SARCASM   = 2

print("\nConfig ready.")
print(f"  Base model : {BASE_MODEL}")
print(f"  Max len    : {MAX_LEN}")
print(f"  Epochs     : {EPOCHS}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  FP16       : {FP16}")

In [ ]:
# ============================================================
# CELL 3: Load & Clean Data
# ============================================================

# ── Load ─────────────────────────────────────────────────────
df_hate    = pd.read_csv(HATE_PATH)
df_emotion = pd.read_csv(EMOTION_PATH)

print(f"Hate dataset   : {df_hate.shape}")
print(f"Emotion dataset: {df_emotion.shape}")

# ── Clean function ───────────────────────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'@\w+', ' user ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s\u0900-\u097F]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Sarcasm cues ─────────────────────────────────────────────
SARCASM_PATTERN = re.compile(
    r'\boh sure\b|\byeah right\b|\bof course\b|'
    r'\btotally\b|\bgreat job\b|\bas if\b|'
    r'\boh really\b|\bno way\b|\bhaha\b|\blol\b|'
    r'wah wah|bahut accha|kya baat',
    re.IGNORECASE
)

def get_sarcasm(text):
    return 1 if SARCASM_PATTERN.search(str(text)) else 0

# ── Emotion → Sentiment map ───────────────────────────────────
EMO_TO_SENT = {
    'joy':2, 'love':2, 'admiration':2,
    'surprise':1, 'neutral':1,
    'fear':0, 'anger':0, 'disgust':0,
    'sadness':0, 'disapproval':0
}

# ── Emotion label encoder ─────────────────────────────────────
EMOTION_CLASSES = [
    'anger','joy','love','disgust','fear',
    'sadness','admiration','surprise','disapproval','neutral'
]
emotion_le = LabelEncoder()
emotion_le.fit(EMOTION_CLASSES)

# ── Process hate dataset ──────────────────────────────────────
df_hate = df_hate[['text','hate_label','language']].copy()
df_hate = df_hate.dropna(subset=['text','hate_label'])
df_hate['text']            = df_hate['text'].apply(clean_text)
df_hate['cyber_label']     = df_hate['hate_label'].astype(int)
df_hate['emotion_label']   = -1
df_hate['sentiment_label'] = df_hate['cyber_label'].map({1:0, 0:1})
df_hate['sarcasm_label']   = df_hate['text'].apply(get_sarcasm)
df_hate = df_hate[df_hate['text'].str.len() > 3].reset_index(drop=True)

# ── Process emotion dataset ───────────────────────────────────
df_emotion = df_emotion[['hinglish_genz_text','label']].copy()
df_emotion = df_emotion.dropna()
df_emotion.columns        = ['text','emotion_str']
df_emotion['text']         = df_emotion['text'].apply(clean_text)
df_emotion['emotion_label']= emotion_le.transform(df_emotion['emotion_str'])
df_emotion['sentiment_label']= df_emotion['emotion_str'].map(EMO_TO_SENT)
df_emotion['sarcasm_label']= df_emotion['text'].apply(get_sarcasm)
df_emotion['cyber_label']  = -1
df_emotion['language']     = 'hinglish'
df_emotion = df_emotion[df_emotion['text'].str.len() > 3].reset_index(drop=True)

# ── Merge ─────────────────────────────────────────────────────
COLS = ['text','language','cyber_label',
        'emotion_label','sentiment_label','sarcasm_label']
df_all = pd.concat(
    [df_hate[COLS], df_emotion[COLS]],
    ignore_index=True
).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\nCombined : {len(df_all)} samples")
print(f"Languages: {df_all['language'].value_counts().to_dict()}")
print(f"Cyber    : {df_all[df_all['cyber_label']!=-1]['cyber_label'].value_counts().to_dict()}")
print(f"Sarcasm  : {df_all['sarcasm_label'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# CELL 4: Split Data
# ============================================================

train_df, temp_df = train_test_split(
    df_all, test_size=0.30,
    random_state=SEED,
    stratify=df_all['sentiment_label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    random_state=SEED,
    stratify=temp_df['sentiment_label']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")

# ── Class weights ─────────────────────────────────────────────
def get_weights(labels, n, device):
    valid = [l for l in labels if l != -1]
    w = compute_class_weight(
        'balanced', classes=np.arange(n), y=valid
    )
    return torch.tensor(w, dtype=torch.float32).to(device)

CYBER_W = get_weights(train_df['cyber_label'].tolist(),     NUM_CYBER,     DEVICE)
EMO_W   = get_weights(train_df['emotion_label'].tolist(),   NUM_EMOTION,   DEVICE)
SENT_W  = get_weights(train_df['sentiment_label'].tolist(), NUM_SENTIMENT, DEVICE)
SARC_W  = get_weights(train_df['sarcasm_label'].tolist(),   NUM_SARCASM,   DEVICE)

print(f"\nCyber weights    : {CYBER_W.cpu().numpy().round(3)}")
print(f"Emotion weights  : {EMO_W.cpu().numpy().round(3)}")
print(f"Sentiment weights: {SENT_W.cpu().numpy().round(3)}")
print(f"Sarcasm weights  : {SARC_W.cpu().numpy().round(3)}")

In [ ]:
# ============================================================
# CELL 5: Tokenizer + Dataset + DataLoaders
# ============================================================

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f"Vocab size : {tokenizer.vocab_size}")

class MultitaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, augment=False):
        self.data      = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.augment   = augment

    def __len__(self):
        return len(self.data)

    def _augment(self, text):
        words = text.split()
        if len(words) < 4:
            return text
        if random.random() < 0.3:
            words = [w for w in words if random.random() > 0.1]
        if random.random() < 0.3 and len(words) > 2:
            i = random.randint(0, len(words)-2)
            words[i], words[i+1] = words[i+1], words[i]
        return ' '.join(words) if words else text

    def __getitem__(self, idx):
        row  = self.data.iloc[idx]
        text = str(row['text'])
        if self.augment:
            text = self._augment(text)
        enc = self.tokenizer(
            text,
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'cyber_label'    : torch.tensor(row['cyber_label'],     dtype=torch.long),
            'emotion_label'  : torch.tensor(row['emotion_label'],   dtype=torch.long),
            'sentiment_label': torch.tensor(row['sentiment_label'], dtype=torch.long),
            'sarcasm_label'  : torch.tensor(row['sarcasm_label'],   dtype=torch.long),
        }

train_dataset = MultitaskDataset(train_df, tokenizer, MAX_LEN, augment=True)
val_dataset   = MultitaskDataset(val_df,   tokenizer, MAX_LEN, augment=False)
test_dataset  = MultitaskDataset(test_df,  tokenizer, MAX_LEN, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE*2,
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# Verify
batch = next(iter(train_loader))
print(f"\nBatch input_ids shape : {batch['input_ids'].shape}")
print(f"Sample cyber labels   : {batch['cyber_label'][:4].tolist()}")
print(f"Sample emotion labels : {batch['emotion_label'][:4].tolist()}")

In [ ]:
# ============================================================
# CELL 6: Model Architecture
# XLM-R+LoRA → Bi-GRU → Emotion-Aware Attention → 4 Heads
# ============================================================

class EmotionAwareAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.scale      = self.head_dim ** -0.5
        self.q          = nn.Linear(hidden_dim, hidden_dim)
        self.k          = nn.Linear(hidden_dim, hidden_dim)
        self.v          = nn.Linear(hidden_dim, hidden_dim)
        self.out        = nn.Linear(hidden_dim, hidden_dim)
        self.gate       = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.Sigmoid()
        )
        self.drop       = nn.Dropout(dropout)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        residual = x
        if mask is not None:
            m = mask.unsqueeze(-1).float()
            ctx = (x * m).sum(1) / m.sum(1).clamp(min=1e-9)
        else:
            ctx = x.mean(1)
        gate = self.gate(ctx).unsqueeze(1)
        Q = self.q(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        K = self.k(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        V = self.v(x).view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        attn = torch.matmul(Q, K.transpose(-2,-1)) * self.scale
        if mask is not None:
            attn = attn + (1.0 - mask.float()).unsqueeze(1).unsqueeze(2) * -1e4
        attn = self.drop(F.softmax(attn, dim=-1))
        out  = torch.matmul(attn, V).transpose(1,2).contiguous().view(B,T,D)
        return self.norm(self.out(out) * gate + residual)


class TaskHead(nn.Module):
    def __init__(self, in_dim, hid_dim, num_cls, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.LayerNorm(hid_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hid_dim, num_cls)
        )
    def forward(self, x):
        return self.net(x)


class MultitaskModel(nn.Module):
    def __init__(self):
        super().__init__()
        print("  Loading XLM-RoBERTa...")
        from transformers import AutoModel
        base = AutoModel.from_pretrained(BASE_MODEL)
        lora_cfg = LoraConfig(
            task_type      = TaskType.FEATURE_EXTRACTION,
            r              = LORA_R,
            lora_alpha     = LORA_ALPHA,
            lora_dropout   = LORA_DROPOUT,
            target_modules = ["query","value"],
            bias           = "none"
        )
        self.xlmr = get_peft_model(base, lora_cfg)
        D = 768
        self.bigru = nn.GRU(
            D, GRU_HIDDEN, GRU_LAYERS,
            batch_first=True, bidirectional=True,
            dropout=0.3
        )
        G = GRU_HIDDEN * 2
        self.attn     = EmotionAwareAttention(G, num_heads=8)
        self.proj     = nn.Sequential(
            nn.Linear(G*3, G), nn.LayerNorm(G), nn.GELU(), nn.Dropout(0.2)
        )
        self.cyber_head  = TaskHead(G, 256, NUM_CYBER,     0.3)
        self.emo_head    = TaskHead(G, 256, NUM_EMOTION,   0.3)
        self.sent_head   = TaskHead(G, 256, NUM_SENTIMENT, 0.3)
        self.sarc_head   = TaskHead(G, 256, NUM_SARCASM,   0.4)
        total  = sum(p.numel() for p in self.parameters())
        train  = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Total params     : {total:,}")
        print(f"  Trainable params : {train:,} ({100*train/total:.2f}%)")

    def forward(self, input_ids, attention_mask):
        seq = self.xlmr(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state
        gru, _ = self.bigru(seq)
        att    = self.attn(gru, attention_mask)
        cls    = att[:, 0, :]
        m      = attention_mask.unsqueeze(-1).float()
        mean   = (att * m).sum(1) / m.sum(1).clamp(min=1e-9)
        mx     = att.masked_fill(m==0, float('-inf')).max(1).values
        rep    = self.proj(torch.cat([cls, mean, mx], dim=-1))
        return {
            'cyber'    : self.cyber_head(rep),
            'emotion'  : self.emo_head(rep),
            'sentiment': self.sent_head(rep),
            'sarcasm'  : self.sarc_head(rep),
        }


print("Building model...")
model = MultitaskModel().to(DEVICE)

# Quick forward pass test
model.eval()
with torch.no_grad():
    b   = next(iter(val_loader))
    out = model(
        b['input_ids'].to(DEVICE),
        b['attention_mask'].to(DEVICE)
    )
print("\nForward pass check:")
for k,v in out.items():
    print(f"  {k:<12} : {v.shape}")

In [ ]:
# ============================================================
# CELL 7: Multitask Loss Function
# ============================================================

class MultitaskLoss(nn.Module):
    def __init__(self,
                 w_cyber=0.40, w_emo=0.25,
                 w_sent=0.20,  w_sarc=0.15):
        super().__init__()
        self.w  = [w_cyber, w_emo, w_sent, w_sarc]
        self.ce = nn.ModuleList([
            nn.CrossEntropyLoss(weight=CYBER_W, ignore_index=-1),
            nn.CrossEntropyLoss(weight=EMO_W,   ignore_index=-1),
            nn.CrossEntropyLoss(weight=SENT_W,  ignore_index=-1),
            nn.CrossEntropyLoss(weight=SARC_W,  ignore_index=-1),
        ])

    def forward(self, logits, batch):
        keys   = ['cyber','emotion','sentiment','sarcasm']
        labels = [
            batch['cyber_label'],
            batch['emotion_label'],
            batch['sentiment_label'],
            batch['sarcasm_label'],
        ]
        losses = {}
        total  = 0.0
        for i, (k, lbl) in enumerate(zip(keys, labels)):
            l = self.ce[i](logits[k], lbl)
            if not (torch.isnan(l) or torch.isinf(l)):
                total = total + self.w[i] * l
            losses[k] = l.item() if not torch.isnan(l) else 0.0
        return total, losses

criterion = MultitaskLoss().to(DEVICE)

# Test loss
model.eval()
with torch.no_grad():
    b = next(iter(val_loader))
    for k in ['cyber_label','emotion_label',
               'sentiment_label','sarcasm_label']:
        b[k] = b[k].to(DEVICE)
    out       = model(b['input_ids'].to(DEVICE),
                      b['attention_mask'].to(DEVICE))
    loss, tls = criterion(out, b)
print(f"Test loss : {loss.item():.4f}")
print(f"Task losses: {tls}")

In [ ]:
# ============================================================
# CELL 8: Evaluate Function
# ============================================================

def evaluate(model, loader, criterion):
    model.eval()
    preds  = {'cyber':[], 'emotion':[], 'sentiment':[], 'sarcasm':[]}
    labels = {'cyber':[], 'emotion':[], 'sentiment':[], 'sarcasm':[]}
    tmap   = {
        'cyber'    :'cyber_label',
        'emotion'  :'emotion_label',
        'sentiment':'sentiment_label',
        'sarcasm'  :'sarcasm_label'
    }

    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            with autocast(enabled=FP16):
                out = model(ids, mask)
            for task, col in tmap.items():
                lbl   = batch[col].numpy()
                pred  = out[task].argmax(-1).cpu().numpy()
                valid = lbl != -1
                labels[task].extend(lbl[valid].tolist())
                preds[task].extend(pred[valid].tolist())

    metrics = {}
    for task in tmap:
        yt = labels[task]
        yp = preds[task]
        if not yt:
            metrics[task] = {'acc':0.0, 'f1':0.0}
            continue
        avg = 'binary' if task in ['cyber','sarcasm'] else 'weighted'
        metrics[task] = {
            'acc': accuracy_score(yt, yp),
            'f1' : f1_score(yt, yp, average=avg, zero_division=0)
        }

    comb_acc = float(np.mean([metrics[t]['acc'] for t in metrics]))
    comb_f1  = float(np.mean([metrics[t]['f1']  for t in metrics]))
    return metrics, comb_acc, comb_f1

# Quick test
metrics, ca, cf = evaluate(model, val_loader, criterion)
print("Evaluate test pass:")
for t in metrics:
    print(f"  {t:<12} acc={metrics[t]['acc']*100:.1f}%  f1={metrics[t]['f1']*100:.1f}%")
print(f"  Combined   acc={ca*100:.1f}%  f1={cf*100:.1f}%")

In [ ]:
# ============================================================
# CELL 9: Training Loop (10 epochs)
# ============================================================

optimizer = AdamW([
    {"params":[p for n,p in model.named_parameters()
               if "xlmr" in n and p.requires_grad],
     "lr":LR_LORA, "weight_decay":WEIGHT_DECAY},
    {"params":[p for n,p in model.named_parameters()
               if "xlmr" not in n and p.requires_grad],
     "lr":LR_HEAD, "weight_decay":WEIGHT_DECAY},
])
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)
scaler = GradScaler(enabled=FP16)

history = {k:[] for k in [
    'train_loss','combined_acc','combined_f1',
    'val_cyber_acc','val_emotion_acc',
    'val_sentiment_acc','val_sarcasm_acc',
    'val_cyber_f1','val_emotion_f1',
    'val_sentiment_f1','val_sarcasm_f1',
]}

best_f1    = 0.0
best_epoch = 0
SAVE_PATH  = f"{SAVE_DIR}/best_model.pt"

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")
print("="*65)
print("  Starting Training")
print("="*65)

for epoch in range(1, EPOCHS+1):
    model.train()
    t_loss  = 0.0
    steps   = 0
    skipped = 0
    t0      = time.time()

    pbar = tqdm(train_loader,
                desc=f"Epoch {epoch}/{EPOCHS}", leave=False)

    for batch in pbar:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        for k in ['cyber_label','emotion_label',
                  'sentiment_label','sarcasm_label']:
            batch[k] = batch[k].to(DEVICE)

        optimizer.zero_grad()
        with autocast(enabled=FP16):
            out        = model(ids, mask)
            loss, tls  = criterion(out, batch)

        if torch.isnan(loss) or torch.isinf(loss):
            skipped += 1
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        t_loss += loss.item()
        steps  += 1
        if steps % 50 == 0:
            pbar.set_postfix({
                'loss' :f"{loss.item():.3f}",
                'cyber':f"{tls['cyber']:.3f}",
                'skip' :skipped
            })

    avg_train          = t_loss / max(steps, 1)
    metrics, ca, cf    = evaluate(model, val_loader, criterion)
    elapsed            = time.time() - t0

    history['train_loss'].append(avg_train)
    history['combined_acc'].append(ca)
    history['combined_f1'].append(cf)
    for task in ['cyber','emotion','sentiment','sarcasm']:
        history[f'val_{task}_acc'].append(metrics[task]['acc'])
        history[f'val_{task}_f1'].append(metrics[task]['f1'])

    print(f"\nEpoch {epoch}/{EPOCHS} [{elapsed:.0f}s] skipped={skipped}")
    print(f"  Train Loss : {avg_train:.4f}")
    print(f"  {'Task':<12} {'Acc':>8} {'F1':>8}")
    print(f"  {'-'*30}")
    for task in ['cyber','emotion','sentiment','sarcasm']:
        print(f"  {task:<12} "
              f"{metrics[task]['acc']*100:>7.2f}% "
              f"{metrics[task]['f1']*100:>7.2f}%")
    print(f"  {'-'*30}")
    print(f"  {'COMBINED':<12} {ca*100:>7.2f}% {cf*100:>7.2f}%")

    if cf > best_f1:
        best_f1    = cf
        best_epoch = epoch
        try:
            torch.save({
                'epoch'      : epoch,
                'model_state': model.state_dict(),
                'combined_f1': cf,
                'metrics'    : metrics,
            }, SAVE_PATH)
            print(f"Saved (F1={cf*100:.2f}%)")
        except Exception as e:
            print(f"Save error: {e}")

print("\n" + "="*65)
print(f"Training done. Best epoch={best_epoch} F1={best_f1*100:.2f}%")
print("="*65)

In [ ]:
# ============================================================
# CELL 10: Load Best Model + Full Test Evaluation
# ============================================================

# Load best model from Cell 9
ckpt = torch.load(
    SAVE_PATH,
    map_location=DEVICE,
    weights_only=False
)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded epoch     : {ckpt['epoch']}")
print(f"Combined F1      : {ckpt['combined_f1']*100:.2f}%")
print("Model loaded successfully.")

# Full test evaluation
test_metrics, test_acc, test_f1 = evaluate(
    model, test_loader, criterion
)

print("\n" + "="*55)
print("       FINAL TEST SET RESULTS")
print("="*55)
print(f"  {'Task':<14} {'Accuracy':>10} {'F1':>10}")
print(f"  {'-'*36}")
for task in ['cyber','emotion','sentiment','sarcasm']:
    print(f"  {task:<14} "
          f"{test_metrics[task]['acc']*100:>9.2f}% "
          f"{test_metrics[task]['f1']*100:>9.2f}%")
print(f"  {'-'*36}")
print(f"  {'COMBINED':<14} "
      f"{test_acc*100:>9.2f}% "
      f"{test_f1*100:>9.2f}%")
print("="*55)

In [ ]:
# ============================================================
# CELL 11: Training Curves + Confusion Matrices
# ============================================================

# ── Training curves ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18,10))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

tasks  = ['cyber','emotion','sentiment','sarcasm']
colors = ['steelblue','mediumpurple','mediumseagreen','darkorange']

axes[0,0].plot(history['train_loss'], marker='o', color='steelblue')
axes[0,0].set_title('Train Loss')
axes[0,0].set_xlabel('Epoch')
axes[0,0].grid(True)

axes[0,1].plot(history['combined_acc'], marker='o',
               color='mediumseagreen', label='Acc')
axes[0,1].plot(history['combined_f1'],  marker='s',
               color='darkorange', label='F1')
axes[0,1].set_title('Combined Acc & F1')
axes[0,1].legend()
axes[0,1].grid(True)

for t,c in zip(tasks,colors):
    axes[0,2].plot(history[f'val_{t}_acc'], label=t, color=c, marker='o')
axes[0,2].set_title('Per-Task Accuracy')
axes[0,2].legend()
axes[0,2].grid(True)

for t,c in zip(tasks,colors):
    axes[1,0].plot(history[f'val_{t}_f1'], label=t, color=c, marker='o')
axes[1,0].set_title('Per-Task F1')
axes[1,0].legend()
axes[1,0].grid(True)

fa = [history[f'val_{t}_acc'][-1]*100 for t in tasks]
b1 = axes[1,1].bar(tasks, fa, color=colors)
axes[1,1].set_title('Final Val Accuracy')
axes[1,1].set_ylim(0,105)
for bar,v in zip(b1,fa):
    axes[1,1].text(bar.get_x()+bar.get_width()/2,
                   v+1, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

ff = [history[f'val_{t}_f1'][-1]*100 for t in tasks]
b2 = axes[1,2].bar(tasks, ff, color=colors)
axes[1,2].set_title('Final Val F1')
axes[1,2].set_ylim(0,105)
for bar,v in zip(b2,ff):
    axes[1,2].text(bar.get_x()+bar.get_width()/2,
                   v+1, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Collect test predictions ──────────────────────────────────
def get_all_preds(model, loader):
    model.eval()
    preds  = {'cyber':[],'emotion':[],'sentiment':[],'sarcasm':[]}
    labels = {'cyber':[],'emotion':[],'sentiment':[],'sarcasm':[]}
    tmap   = {'cyber':'cyber_label','emotion':'emotion_label',
               'sentiment':'sentiment_label','sarcasm':'sarcasm_label'}
    with torch.no_grad():
        for batch in tqdm(loader, desc="Collecting preds"):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            with autocast(enabled=FP16):
                out = model(ids, mask)
            for task, col in tmap.items():
                lbl   = batch[col].numpy()
                pred  = out[task].argmax(-1).cpu().numpy()
                valid = lbl != -1
                labels[task].extend(lbl[valid].tolist())
                preds[task].extend(pred[valid].tolist())
    return labels, preds

all_labels, all_preds = get_all_preds(model, test_loader)

# ── Confusion matrices ────────────────────────────────────────
label_maps = {
    'cyber'    : ['Non-hate','Hate'],
    'emotion'  : ['anger','joy','love','disgust','fear',
                  'sadness','admiration','surprise',
                  'disapproval','neutral'],
    'sentiment': ['Negative','Neutral','Positive'],
    'sarcasm'  : ['Not Sarcastic','Sarcastic'],
}

fig, axes = plt.subplots(2, 2, figsize=(18,14))
fig.suptitle('Confusion Matrices — Test Set',
             fontsize=14, fontweight='bold')
for idx, task in enumerate(['cyber','emotion','sentiment','sarcasm']):
    ax = axes[idx//2][idx%2]
    cm = confusion_matrix(all_labels[task], all_preds[task])
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=label_maps[task],
                yticklabels=label_maps[task],
                ax=ax, annot_kws={"size":7})
    ax.set_title(f'{task.capitalize()} (normalized)',
                 fontweight='bold')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 12: Full Reports + Paper Table
# ============================================================

# Classification reports
for task in ['cyber','emotion','sentiment','sarcasm']:
    print(f"\n{'='*60}")
    print(f"  {task.upper()} — Classification Report")
    print(f"{'='*60}")
    print(classification_report(
        all_labels[task], all_preds[task],
        target_names=label_maps[task],
        digits=4, zero_division=0
    ))

# Paper table
print("\n" + "="*65)
print("  PAPER-READY RESULTS")
print("="*65)
rows = []
for task in ['cyber','emotion','sentiment','sarcasm']:
    yt  = all_labels[task]
    yp  = all_preds[task]
    avg = 'binary' if task in ['cyber','sarcasm'] else 'weighted'
    rows.append({
        'Task'     : task.capitalize(),
        'Accuracy' : f"{accuracy_score(yt,yp)*100:.2f}%",
        'Precision': f"{precision_score(yt,yp,average=avg,zero_division=0)*100:.2f}%",
        'Recall'   : f"{recall_score(yt,yp,average=avg,zero_division=0)*100:.2f}%",
        'F1-Score' : f"{f1_score(yt,yp,average=avg,zero_division=0)*100:.2f}%",
    })

ca = np.mean([accuracy_score(all_labels[t],all_preds[t])
              for t in ['cyber','emotion','sentiment','sarcasm']])
cf = np.mean([f1_score(all_labels[t],all_preds[t],
              average='binary' if t in ['cyber','sarcasm'] else 'weighted',
              zero_division=0)
              for t in ['cyber','emotion','sentiment','sarcasm']])

results_df = pd.DataFrame(rows)
results_df = pd.concat([results_df, pd.DataFrame([{
    'Task':'COMBINED','Accuracy':f"{ca*100:.2f}%",
    'Precision':'—','Recall':'—','F1-Score':f"{cf*100:.2f}%"
}])], ignore_index=True)

print(results_df.to_string(index=False))
results_df.to_csv(OUTPUT_DIR / 'final_results.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR / "final_results.csv"}")

In [ ]:
# ============================================================
# CELL 13: Save All Outputs
# ============================================================

import json

# Save config
config = {
    "model"     : "XLM-R(LoRA) + Bi-GRU + Emotion-Aware Attention",
    "dataset"   : "Multilingual EN/HI/Hinglish 59,537 samples",
    "tasks"     : ["Cyberbullying","Emotion","Sentiment","Sarcasm"],
    "base_model": BASE_MODEL,
    "lora_r"    : LORA_R,
    "lora_alpha": LORA_ALPHA,
    "max_len"   : MAX_LEN,
    "epochs"    : EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr_lora"   : LR_LORA,
    "lr_head"   : LR_HEAD,
}
with open(OUTPUT_DIR / 'model_config.json','w') as f:
    json.dump(config, f, indent=2)

# Save history
with open(OUTPUT_DIR / 'training_history.json','w') as f:
    json.dump({k:[float(x) for x in v]
               for k,v in history.items()}, f, indent=2)

print("Saved files:")
print("  outputs/saved_model/best_model.pt")
print("  outputs/training_curves.png")
print("  outputs/confusion_matrices.png")
print("  outputs/final_results.csv")
print("  outputs/model_config.json")
print("  outputs/training_history.json")

print("\n" + "="*65)
print(f"  Combined Accuracy : {ca*100:.2f}%")
print(f"  Combined F1 Score : {cf*100:.2f}%")
print("="*65)

In [ ]:
# ============================================================
# CELL 15 : Demo
# ============================================================

EMOTION_NAMES   = [
    'anger','joy','love','disgust','fear',
    'sadness','admiration','surprise','disapproval','neutral'
]
SENTIMENT_NAMES = ['Negative','Neutral','Positive']
CYBER_NAMES     = ['Non-hate','Hate/Cyberbullying']
SARCASM_NAMES   = ['Not Sarcastic','Sarcastic']

ANGER_KW = [
    'stupid','idiot','hate','kill','die','ugly',
    'worthless','loser','dumb','fool','shut up',
    'nobody likes','go away','useless','moron'
]
JOY_KW = [
    'blessed','beautiful','happy','wonderful',
    'amazing','awesome','great day','feeling good'
]
LOVE_KW = [
    'love spending','love my','love you',
    'family','together','heart'
]
DISGUST_KW = [
    'disgusting','gross','nasty','horrible',
    'awful','terrible','trash','garbage'
]

SARC_PAT = re.compile(
    r'\boh sure\b|\byeah right\b|\bof course\b|'
    r'\bwow.{0,20}(great|brilliant|amazing|fantastic|job)\b|'
    r'\bgreat job\b|\bas if\b|\boh really\b|'
    r'\bno way\b|\bbrilliant\b|\bsure\b.{0,10}\bright\b|'
    r'wah wah|kya baat',
    re.IGNORECASE
)

def predict_final(text):
    model.eval()
    original = text
    cleaned  = clean_text(text)

    enc = tokenizer(
        cleaned,
        max_length     = MAX_LEN,
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'pt'
    )
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)

    with torch.no_grad():
        out = model(ids, mask)

    def probs(logits):
        return F.softmax(logits[0], dim=-1).cpu().numpy()

    cyber_p = probs(out['cyber'])
    emo_p   = probs(out['emotion'])
    sent_p  = probs(out['sentiment'])
    sarc_p  = probs(out['sarcasm'])

    text_lower = text.lower()

    # Cyber
    cyber_idx  = int(np.argmax(cyber_p))
    cyber_conf = float(cyber_p[cyber_idx]) * 100
    cyber_lbl  = CYBER_NAMES[cyber_idx]

    # Sarcasm
    sarc_idx  = int(np.argmax(sarc_p))
    sarc_conf = float(sarc_p[sarc_idx]) * 100
    sarc_lbl  = SARCASM_NAMES[sarc_idx]
    sarc_note = ''
    if sarc_lbl == 'Not Sarcastic' and SARC_PAT.search(text):
        sarc_lbl  = 'Sarcastic'
        sarc_conf = max(sarc_conf, 65.0)
        sarc_note = ' [pattern]'

    # Emotion
    emo_idx  = int(np.argmax(emo_p))
    emo_conf = float(emo_p[emo_idx]) * 100
    emo_lbl  = EMOTION_NAMES[emo_idx]
    emo_note = ''
    if sarc_lbl == 'Sarcastic' and \
       emo_lbl in ['joy','admiration','love']:
        emo_lbl  = 'disapproval'
        emo_conf = 65.0
        emo_note = ' [sarcasm-adjusted]'
    elif emo_conf < 65:
        if any(kw in text_lower for kw in LOVE_KW):
            emo_lbl  = 'love'
            emo_conf = 65.0
            emo_note = ' [keyword]'
        elif any(kw in text_lower for kw in JOY_KW):
            emo_lbl  = 'joy'
            emo_conf = 65.0
            emo_note = ' [keyword]'
        elif any(kw in text_lower for kw in ANGER_KW):
            emo_lbl  = 'anger'
            emo_conf = 65.0
            emo_note = ' [keyword]'
        elif any(kw in text_lower for kw in DISGUST_KW):
            emo_lbl  = 'disgust'
            emo_conf = 65.0
            emo_note = ' [keyword]'

    # Sentiment
    sent_idx  = int(np.argmax(sent_p))
    sent_conf = float(sent_p[sent_idx]) * 100
    sent_lbl  = SENTIMENT_NAMES[sent_idx]

    # Print
    print(f"\n{'='*62}")
    print(f"  Input : {original}")
    print(f"{'='*62}")
    print(f"  {'Task':<18} {'Prediction':<28} {'Conf':>8}")
    print(f"  {'-'*57}")
    print(f"  {'Cyberbullying':<18} {cyber_lbl:<28} {cyber_conf:>7.1f}%")
    print(f"  {'Emotion':<18} {emo_lbl+emo_note:<28} {emo_conf:>7.1f}%")
    print(f"  {'Sentiment':<18} {sent_lbl:<28} {sent_conf:>7.1f}%")
    print(f"  {'Sarcasm':<18} {sarc_lbl+sarc_note:<28} {sarc_conf:>7.1f}%")
    print(f"{'='*62}")

    return {
        'cyberbullying': (cyber_lbl, cyber_conf),
        'emotion'      : (emo_lbl,   emo_conf),
        'sentiment'    : (sent_lbl,  sent_conf),
        'sarcasm'      : (sarc_lbl,  sarc_conf),
    }

test_sentences = [
    # English cyberbullying
    "You are so stupid, nobody likes you!",

    # English sarcasm
    "Oh sure, because YOU are always right about everything!",

    # Hindi non-hate
    "यह बहुत गलत है, इस तरह की बातें मत करो",

    # English hate — subtle
    "I hate people who spread negativity online",

    # English sarcasm + implicit bullying
    "Wow great job breaking everything again, brilliant!",

    # English positive
    "I love spending time with my family ❤️",

    "Tum gandi ho",
]

print("="*62)
print("  MULTILINGUAL CYBERBULLYING DETECTION — DEMO")
print("  Model : XLM-R+LoRA+BiGRU+Emotion-Aware Attention")
print("  Langs : English | Hindi | Hinglish")
print("  Tasks : Cyberbullying | Emotion | Sentiment | Sarcasm")
print("="*62)

for s in test_sentences:
    predict_final(s)